# KEEP graph closure analysis: connectivity, mapping coverage, and learning limits

Filtered direct ontology relationships and separation-one ancestry pairs can yield different traversal graphs. We investigated whether PyHealth's closure option changes useful connectivity and mapping coverage, and whether those observations say anything about the representations supplied to learning.

**Core finding:** separation-one ancestry augmentation can improve traversal-graph connectivity and mapping coverage on the measured Athena bundle. **Its effect on embedding quality or downstream learning is inconclusive with the current output.** More reachable concepts and paths do not by themselves establish better representations.

PyHealth's existing `path_source="closure"` option provides the augmentation capability. This contribution adds a reproducible analysis of that behavior; it does not introduce the option or change production defaults. The notebook begins with executable synthetic examples. Recorded full-release findings are presented separately; rerunning real input requires an explicit Athena path.


## What closure means here

Let `U` be the uncut subtree of valid standard SNOMED Condition concepts under Disease (OMOP concept `4274025`). `D` contains valid direct `Is a` edges with both endpoints in `U`, oriented parent → child. `P` contains unique ancestry pairs with minimum separation **exactly one**, with both endpoints in `U`.

- Direct mode traverses `D`; closure mode traverses `D ∪ P`. Direct provenance wins on overlap.
- Athena minimum root separations remain authoritative in both modes. The target cutoff does not truncate deep sources or intermediate nodes.
- The primary comparison fixes cutoff five, excludes the root, disables orphan rescue, and runs both `first_retained` and `all_retained`. Neither target policy is author-verified here.
- This overlay is distinct from arbitrary transitive closure or longer orphan-rescue shortcuts. OHDSI describes ancestry as a derived table; it should not be assumed identical to a consumer's filtered direct subgraph. See the [OMOP ancestry table definition](https://ohdsi.github.io/CommonDataModel/cdm54.html#concept_ancestor).

Three graph layers matter: the factory's **base KnowledgeGraph remains direct** in both modes; the processor's **traversal index** gains the overlay; generated **sample graphs** contain the union of successful mapping paths and may therefore change. The Athena export itself is not modified.

In [ ]:
from pathlib import Path
import sys
import tempfile

repo = Path.cwd() if (Path.cwd() / 'pyhealth').is_dir() else Path.cwd().parent
assert (repo / 'pyhealth').is_dir(), 'Start Jupyter in the repository or examples directory'
sys.path.insert(0, str(repo))
from examples.keep_closure_mode_audit import write_synthetic_bundle, run_audit
from pyhealth.medcode import OMOPVocabulary
from pyhealth.tasks import build_keep_omop_ancestor_rollup

temporary = tempfile.TemporaryDirectory()
folder = Path(temporary.name) / 'synthetic'
write_synthetic_bundle(folder)
summary = run_audit(folder, Path(temporary.name) / 'audit', synthetic=True)
summary['graph_counts']

## Synthetic expectations

Twelve Condition concepts are in U, with six eligible targets and five deep sources. Two added edges have different effects: `104 → 201` recovers an orphan; `102 → 204` adds a first-retained target already reachable under all-retained. A filtered Observation between `104` and `202` needs a longer shortcut, so separation-one overlay cannot repair that mapping.

These tables intentionally stress authoritative depths, including disagreement with adjacency distances. Their hand-authored ancestry rows are a semantic fixture, not a simulated complete Athena release.

In [ ]:
assert summary['graph_counts'] == dict(U=12, D=9, P=11, intersection=9, P_minus_D=2, D_minus_P=0, union=11)
for policy, result in summary['policies'].items():
    population = result['populations']['all']
    assert population['denominator'] == 12
    assert population['direct_no_target'] == 3
    assert population['closure_no_target'] == 2
    assert population['categories']['recovered_no_target'] == 1
    print(policy, population['categories'])

## Branch stopping, graph gaps, and failures

Both policies preserve old targets when acyclic edges are only added at fixed eligibility. An existing path to a first-retained target still exists with no newly eligible internal node.

A disconnected eligible source (`203`) maps to itself; a deep source (`202`) stays `no_target`. The excluded root is also `no_target` and is counted separately from deep failures. Diagnostic enumeration uses `on_unmapped="empty"`; the production default is `"error"`.

In [ ]:
vocabulary = OMOPVocabulary(folder)
for mode in ('direct', 'closure'):
    processor = build_keep_omop_ancestor_rollup(vocabulary, target_policy='first_retained', include_root=False, path_source=mode, on_unmapped='empty')
    rows = processor.rollup_codes(['201', '202', '203', '204'])
    print(mode, [(r.source, r.targets, r.reason) for r in rows])
    assert rows[1].reason == 'no_target'
    assert rows[2].targets == ('203',)
    eligible = build_keep_omop_ancestor_rollup(vocabulary, target_policy='first_retained', include_root=False, path_source=mode, max_depth=7, on_unmapped='empty')
    assert eligible.rollup_codes(['202'])[0].targets == ('202',)
assert summary['policies']['first_retained']['populations']['all']['categories']['changed_nonempty_targets'] == 2
assert summary['policies']['all_retained']['populations']['all']['categories']['changed_nonempty_targets'] == 0

## Recorded Athena findings — distinct from the synthetic execution

Measured on 2026-11-25 using the user-supplied export. These results cover every concept in that bundle's uncut Disease subtree, not every vocabulary available through Athena. They are recorded observations, not outputs of the synthetic cells above.

The bundle identifies OMOP Standardized Vocabularies as `v5.0 29-AUG-26`. Its SNOMED version is International `2026-02-01`, US `2026-03-01`, and UK `2026-04-08`. The `Vocabulary` metadata row has no version; modification/download dates are not inferred release identities.

| Graph quantity | Count |
| --- | ---: |
| Concepts in U | 92,676 |
| Eligible targets, both modes | 69,445 |
| Deep sources, depth > 5 | 23,230 |
| Excluded root | 1 |
| Direct edges D | 206,783 |
| Separation-one pairs P | 207,117 |
| D ∩ P | 206,783 |
| P − D: closure-only edges | 334 |
| D − P | 0 |
| D ∪ P | 207,117 |

Factory provenance agrees with the reference counts: `closure_only_edge_count = 334 = |P − D|`. The additional edges are 0.162% of the direct edge count; that fraction alone does not establish their importance.

| Structural metric | Direct | Closure |
| --- | ---: | ---: |
| Non-root nodes without an incoming parent edge | 36 | 11 |
| Of those: eligible / deep | 23 / 13 | 8 / 3 |
| Nodes unreachable from Disease along selected edges | 190 | 133 |
| Of those: eligible / deep | 77 / 113 | 46 / 87 |

These counts establish improved structural connectivity under the tested settings, not guaranteed connectivity. An eligible disconnected source still maps to itself; a deep source can reach an eligible target in a branch disconnected from Disease. Structural gaps and mapping failures therefore need separate measurements.

## Mapping coverage and failure rates

Every source in U was evaluated once per mode and target policy, in deterministic batches with the same depths and eligible targets. Diagnostic `on_unmapped="empty"` preserves typed failures for enumeration; production defaults to `"error"`.

| Outcome | first_retained | all_retained |
| --- | ---: | ---: |
| All-U denominator | 92,676 | 92,676 |
| Identical target sets, all U | 92,624 | 92,611 |
| Changed nonempty target sets | 26 | 39 |
| Direct no_target recovered by closure | 26 | 26 |
| Any changed target set | 52 | 65 |
| Lost mappings | 0 | 0 |
| Identity mappings, either mode | 69,445 | 69,445 |
| Direct / closure rolled-up mappings | 23,197 / 23,223 | 23,197 / 23,223 |
| Direct / closure unmapped, all U | 34 / 8 | 34 / 8 |
| Deep denominator | 23,230 | 23,230 |
| Direct / closure no_target, deep | 33 / 7 | 33 / 7 |

Both policies recover **26 of 33 deep failures (78.8%)**. The deep mapping failure rate falls from **0.142% to 0.030%**, an absolute decrease of **0.112 percentage points**. Seven deep failures remain. All-U failures additionally include the expected excluded-root `no_target`; external-code failure reasons are zero within this deliberately restricted U population.

Use **mapping failure rate**, not semantic error rate: finding an ancestor establishes reachability, not independent clinical correctness. Concept-level rates are not diagnosis-event or patient-level rates. All observed ordered-target changes also change sets. Fixed-eligibility target sets are preserved when edges are added under both policies.

A separate diagnostic scan found eligible ancestry for all seven residual deep failures: nearest eligible separations were three for three sources, four for three, and five for one. There is still no selected traversal path to those targets. This demonstrates the difference between separation-one augmentation and longer ancestry lookup; it does not establish that the existing orphan-rescue algorithm would repair all seven.

## What this means for generated graphs and embeddings

Closure can change the representation supplied to learning in three ways:

1. **Coverage:** previously unmapped observations can contribute a graph.
2. **Conceptual scope:** additional targets or intermediate concepts can become represented.
3. **Topology:** additional successful paths can change graph input even when target sets stay identical.

For example, direct traversal might produce `T → A → X`, while augmentation adds `T → X`. Both modes still map X to retained target T, yet their generated graphs differ. Depending on edge direction and architecture, the shortcut changes neighborhoods, degrees and the number of message-passing steps between concepts. For a GCN, adjacency and degree normalization directly enter the computation; see the [original GCN paper](https://arxiv.org/abs/1609.02907).

Greater connectivity can restore useful context, but additional shared context could also make representations less discriminative. Those are hypotheses. **Current output establishes structural and coverage changes, not improved embeddings.** A separate embedding trainer consuming only the unchanged base KnowledgeGraph would not automatically receive the overlay.

The all-U mapping census measures recovery and target changes, with selected projected-path witnesses. It does not yet measure the full population's generated-graph changes, diagnosis-weighted exposure, trained embeddings or predictive performance. A small vocabulary-edge delta may affect common sample paths, or rarely occur in the intended data; we have not measured which. Retraining shared parameters could also change predictions for samples whose own graphs remain unchanged.

The supported conclusion is to retain an explicit augmentation capability based on demonstrated connectivity and coverage on this bundle, while leaving defaults, orphan rescue and embedding-training graph choices open. This is not evidence of author-artifact parity, clinical benefit or a universally better graph representation.

## Methodological extensions, not completed results

The contribution is a reproducible validation workflow for a consequential graph-preprocessing choice. The next evidence should explain what information and structure closure introduces into learning input.

| Priority | Next measurement | Purpose |
| --- | --- | --- |
| Generated graph scope | Stream paired `process([source])` outputs across U; compare decoded node/edge sets, targets, seeds, masks and size distributions; cross-tabulate graph changes against target changes | Detect topology changes hidden by identical mappings |
| Independent correctness | Test-only path enumeration and independent table extraction on coherent synthetic fixtures; validate multi-source union graphs, feature alignment, duplicate handling and optional batch/unbatch behavior | Reduce dependence on shared reader/traversal logic |
| Sensitivity | Matched cutoff sweep 3–7 for both policies with recomputed eligible/deep denominators; controlled missing-link and filtered-intermediate fixtures | Identify dependence on eligibility and the limits of recovery |
| Exposure and learning | Authorized sample-level exposure counts, then a separately scoped matched learning experiment | Determine practical frequency and whether representation changes improve a task |

Compare decoded concept IDs and edges, not local indices or fingerprints as substitutes for graph contents. An edgeless identity graph is valid; an empty graph for an unmapped source is a coverage outcome rather than a malformed graph. Separate graph-contract violations, construction exceptions and mapping failures. Keep synthetic composed samples separate from claims about patient frequencies.

The current factory and reference share the public OMOP reader, so their agreement cannot exclude a shared extraction bug. An independent synthetic reference and table extraction strengthen the method. A second authorized release would test replication; the current census does not establish cross-release robustness. Report exact counts rather than treating correlated ontology concepts as independent random samples for confidence intervals.

A future learning comparison should retain identical cohorts and splits, matched initial embeddings/model settings, paired training seeds and an explicit empty/unmapped-sample policy. First isolate generated-graph effects; retraining embeddings on alternate graphs is another intervention. A frozen-model input swap measures sensitivity, not benefit from retraining. Selective sample dropping must not masquerade as improved prediction. None of these learning experiments has been performed.

## Reproducibility and validation record

The measured production baseline is `33c2cb635572a493dae428c4666a5ec503c5ee77`, before the separate loading optimization. The subsequently published helper/notebook/test commit is `b2a5150`; this notebook's narrative is updated after that checkpoint. The measured local JSON records exact source hashes, input metadata, configuration, dirty state and processor fingerprints. Do not attribute these measurements to the later loader optimization.

| Input table | Bytes | SHA-256 |
| --- | ---: | --- |
| CONCEPT | 203,864,313 | `5be10a0d8900083c9b56cf2e53f17a7e15195879184615512c03690a98da65a4` |
| CONCEPT_RELATIONSHIP | 433,313,452 | `aa2f23b52c5d5f97519b6176469087dad965b7838e73eb62e18160e3696736b8` |
| CONCEPT_ANCESTOR | 174,318,785 | `5e97d2d88d0ffac721e914e683d2af8c92c57721b0415f03377bc9c9f67a29e5` |
| VOCABULARY | 3,710 | `1f5e1fa97c8be54d9f5afeb7d673d5040952c0e9318db125957aa71209e7eb6a` |

Python 3.12.12, PyHealth 2.1.0 from the checkout, Polars 1.35.2, PyTorch 2.15.0, NetworkX 3.7 and torch-geometric 2.8.1 were used on macOS arm64 with 18 reported logical CPUs.

The helper writes aggregate `summary.json`, per-policy `*_differences.tsv` and `*_failures.tsv`, full ordered mapping digests and up to three path witnesses per category. Difference tables contain changed sources; failure tables retain every failed source, including the excluded root. Absent aggregate reason keys mean zero. Keep licensed input and release-specific rows local.

The original focused and existing OMOP/roll-up checks passed **140 tests and 8 subtests**; the synthetic notebook ran in a fresh kernel with saved outputs cleared. Two full-release runs with batch sizes 512 and 137 reproduced non-resource summaries, processor fingerprints, ordered mapping digests and byte-identical difference/failure tables. Timing, memory, dirty-state text and batch-size fields were excluded from that comparison.

The public reference reader and this baseline factory materialize full uncut ancestry. Main-run reference extraction took 0.447 s; direct/closure builds took 1.039/1.118 s for first-retained and 1.030/1.114 s for all-retained. Full mapping took 0.103/0.208 s and 0.296/0.401 s respectively. Lifetime process peak RSS was 2,217,984,000 bytes (about 2.07 GiB), measured with `getrusage(RUSAGE_SELF)` across all phases. Input hashing, assertions, serialization and witness projection are outside mapping timings. Concurrent validation work makes these resource context, not controlled speed comparisons; full projected-graph costs have not been measured.

Run the default synthetic example from the repository root with `python -m examples.keep_closure_mode_audit`. For a real release, pass `--athena-dir /path/to/authorized/export --output-dir .artifacts/graph-closure-analysis`. No dataset discovery or download occurs. The opt-in cell below uses the same helper; the recorded release findings above do not imply that cell ran in this notebook execution.

## Explicit opt-in release audit

Set `ATHENA_DIR` to an authorized local export and choose `LOCAL_RESULTS` outside shared artifacts. No path discovery or automatic fallback occurs. The helper evaluates every concept once per mode and policy in batches. It records release metadata, table hashes, failures, differences and small path witnesses. Full ancestor tables are materialized by the public reference reader and this baseline factory; this is included in resource reporting.

Release-specific rows remain local. Vocabulary behavior does not establish diagnosis prevalence, clinical benefit, embedding quality, or fidelity to unpublished author artifacts.

In [ ]:
ATHENA_DIR = None  # Explicit Path('/your/authorized/athena-export') to opt in.
LOCAL_RESULTS = repo / '.artifacts' / 'graph-closure-analysis-athena'
if ATHENA_DIR is not None:
    real_summary = run_audit(Path(ATHENA_DIR), LOCAL_RESULTS, synthetic=False)
    print(real_summary['graph_counts'])
else:
    print('Athena execution skipped; synthetic results only in this execution.')

In [ ]:
temporary.cleanup()